<a href="https://colab.research.google.com/github/Vaibhav77e/train-custom-yolo-model/blob/main/train_yolo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.7 MB/s eta 0:00:00


In [7]:
with open("/content/dataset/data.yaml", "w") as f:
    f.write("""
path: /content/dataset

train: images/train
val: images/val

names:
  0: table_without_border
  1: table_with_border
""")

In [3]:
!apt-get install unrar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


In [4]:
!unrar x dataset.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from dataset.rar

Creating    dataset                                                   OK
Extracting  dataset/data.yaml                                              0%  OK 
Creating    dataset/images                                            OK
Creating    dataset/images/train                                      OK
Extracting  dataset/images/train/100vegan.jpg                              0%  OK 
Extracting  dataset/images/train/101vegan.jpg                              0%  OK 
Extracting  dataset/images/train/102vegan.jpg                              0%  OK 
Extracting  dataset/images/train/104vegan.jpg                              0%  OK 
Extracting  dataset/images/train/105vegan.jpg                              0%  OK 
Extracting  dataset/images/train/106vegan.jpg                              1%  OK 
Extracting  dataset/images/train

In [8]:
print(open("/content/dataset/data.yaml").read())


path: /content/dataset

train: images/train
val: images/val

names:
  0: table_without_border
  1: table_with_border



In [9]:
import os
import shutil
import random

train_img = "/content/dataset/images/train"
val_img = "/content/dataset/images/val"
train_lbl = "/content/dataset/labels/train"
val_lbl = "/content/dataset/labels/val"

images = os.listdir(train_img)
random.shuffle(images)

split = int(len(images) * 0.2)  # 20% validation
val_images = images[:split]

for img in val_images:
    shutil.move(f"{train_img}/{img}", f"{val_img}/{img}")

    label = img.replace(".jpg", ".txt").replace(".png", ".txt")
    if os.path.exists(f"{train_lbl}/{label}"):
        shutil.move(f"{train_lbl}/{label}", f"{val_lbl}/{label}")

print("Validation split created")

Validation split created


In [10]:
import os
print("Train images:", len(os.listdir("/content/dataset/images/train")))
print("Val images:", len(os.listdir("/content/dataset/images/val")))

Train images: 336
Val images: 84


In [ ]:
from ultralytics import YOLO

model = YOLO("yolo26m.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=80,
    imgsz=1024,
    device=0
)

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7be5dbf81bb0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04804

In [19]:
from ultralytics import YOLO

# load trained model
model = YOLO("/content/runs/detect/train4/weights/best.pt")

# run prediction
model.predict(
    source="/content/test_image_4.png",
    conf=0.4,
    save=True
)


image 1/1 /content/test_image_4.png: 736x1024 1 table_with_border, 11.8ms
Speed: 6.0ms preprocess, 11.8ms inference, 1.3ms postprocess per image at shape (1, 3, 736, 1024)
Results saved to /content/runs/detect/predict4


[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'table_without_border', 1: 'table_with_border'}
 obb: None
 orig_img: array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        ...,
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
         [0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0],
         ...,
     

In [27]:
!unrar x processed_images_with_black_background2.rar


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from processed_images_with_black_background2.rar

Creating    processed_images_with_black_background2                   OK
Extracting  processed_images_with_black_background2/17. L25_06827_01-1-0506-28109_page_1_clean.png       0%  1%  OK 
Extracting  processed_images_with_black_background2/18. L25_06827_02-1-0506-95c44_page_1_clean.png       1%  2%  3%  OK 
Extracting  processed_images_with_black_background2/19. L25_06828_01-1-0505-e23c0_page_1_clean.png       3%  4%  5%  OK 
Extracting  processed_images_with_black_background2/20. L26_00070_01-1-0107-2c737_page_1_clean.png       5%  6%  OK 
Extracting  processed_images_with_black_background2/21. L26_00071_01-1-0107-806a1_page_1_clean.png       6%  OK 
Extracting  processed_images_with_black_background2/22. L26_00099_01-1-0107-a076a_page_1_clean.png       7%  8%  OK 
Ex

In [ ]:
from ultralytics import YOLO
import os

# load trained model
model = YOLO("/content/runs/detect/train4/weights/best.pt")

# input folder
input_folder = "/content/processed_images_with_black_background2"

# output folder
output_folder = "/content/table_detection_results"

os.makedirs(output_folder, exist_ok=True)

# run prediction on entire folder
model.predict(
    source=input_folder,
    conf=0.4,
    save=True,
    project=output_folder,   # custom save location
    name="results",          # subfolder name
    device=0
)

In [28]:
# from ultralytics import YOLO
# import os
# import time

# # load trained model
# print("Loading YOLO model...")
# model = YOLO("/content/runs/detect/train4/weights/best.pt")
# print("Model loaded successfully!\n")

# # input folder
# input_folder = "/content/processed_images_with_black_background2"

# # output folder
# output_folder = "/content/table_detection_results"
# os.makedirs(output_folder, exist_ok=True)

# print(f"Input folder: {input_folder}")
# print(f"Output folder: {output_folder}")

# # collect images
# images = [f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp'))]

# print(f"Total images found: {len(images)}")
# print("Starting detection...\n")

# total_start = time.time()

# for i, img in enumerate(images):

#     image_path = os.path.join(input_folder, img)

#     print(f"[{i+1}/{len(images)}] Processing: {img}")

#     start = time.time()

#     model.predict(
#         source=image_path,
#         conf=0.4,
#         save=True,
#         project=output_folder,
#         name="results",
#         device=0,
#         verbose=False
#     )

#     end = time.time()

#     print(f"Time taken: {round(end-start,2)} seconds\n")

# total_end = time.time()

# print("All images processed successfully!")
# print(f"Total time: {round(total_end-total_start,2)} seconds")
# print(f"Results saved in: {output_folder}/results/")

In [30]:
# import os
# import shutil

# folder = "/content/table_detection_results"

# if os.path.exists(folder):
#     shutil.rmtree(folder)
#     print("Folder deleted successfully")
# else:
#     print("Folder does not exist")

from ultralytics import YOLO
import os
import cv2
import time

print("Loading model...")
model = YOLO("/content/runs/detect/train4/weights/best.pt")
print("Model loaded!\n")

input_folder = "/content/processed_images_with_black_background2"
output_folder = "/content/table_detection_results"

os.makedirs(output_folder, exist_ok=True)

images = [f for f in os.listdir(input_folder) if f.lower().endswith(('.jpg','.png','.jpeg'))]

print(f"Total images: {len(images)}\n")

for i, img_name in enumerate(images):

    start = time.time()

    img_path = os.path.join(input_folder, img_name)

    print(f"[{i+1}/{len(images)}] Processing {img_name}")

    results = model(img_path)

    # draw bounding boxes
    annotated = results[0].plot()

    # save directly to output folder
    save_path = os.path.join(output_folder, img_name)
    cv2.imwrite(save_path, annotated)

    end = time.time()

    print(f"Saved → {save_path}")
    print(f"Time: {round(end-start,2)} sec\n")

print("All images processed!")

Loading model...
Model loaded!

Total images: 102

[1/102] Processing 24. L26_00102_01-1-0107-bc2a1_page_2_clean.png

image 1/1 /content/processed_images_with_black_background2/24. L26_00102_01-1-0107-bc2a1_page_2_clean.png: 1024x736 1 table_with_border, 11.7ms
Speed: 5.5ms preprocess, 11.7ms inference, 1.2ms postprocess per image at shape (1, 3, 1024, 736)
Saved → /content/table_detection_results/24. L26_00102_01-1-0107-bc2a1_page_2_clean.png
Time: 0.5 sec

[2/102] Processing 64. L26_00156_07-1-0108-192e4_page_1_clean.png

image 1/1 /content/processed_images_with_black_background2/64. L26_00156_07-1-0108-192e4_page_1_clean.png: 736x1024 1 table_without_border, 11.9ms
Speed: 7.0ms preprocess, 11.9ms inference, 1.9ms postprocess per image at shape (1, 3, 736, 1024)
Saved → /content/table_detection_results/64. L26_00156_07-1-0108-192e4_page_1_clean.png
Time: 0.89 sec

[3/102] Processing Sample14-1_page_2_clean.png

image 1/1 /content/processed_images_with_black_background2/Sample14-1_pag

In [31]:
# compress folder
# !zip -r table_detection_results.zip /content/table_detection_results

# # download
# from google.colab import files
# files.download("table_detection_results.zip")

  adding: content/table_detection_results/ (stored 0%)
  adding: content/table_detection_results/24. L26_00102_01-1-0107-bc2a1_page_2_clean.png (deflated 3%)
  adding: content/table_detection_results/64. L26_00156_07-1-0108-192e4_page_1_clean.png (deflated 3%)
  adding: content/table_detection_results/Sample14-1_page_2_clean.png (deflated 5%)
  adding: content/table_detection_results/33. L26_00117_01-1-0108-13411_page_4_clean.png (deflated 3%)
  adding: content/table_detection_results/Sample13-1_page_3_clean.png (deflated 5%)
  adding: content/table_detection_results/45. L26_00139_01-1-0108-f617f_page_1_clean.png (deflated 3%)
  adding: content/table_detection_results/Sample11-1_page_2_clean.png (deflated 4%)
  adding: content/table_detection_results/Sample16-1_page_1_clean.png (deflated 2%)
  adding: content/table_detection_results/Sample7-1_page_4_clean.png (deflated 6%)
  adding: content/table_detection_results/61. L26_00156_04-2-0108-30c3f_page_1_clean.png (deflated 3%)
  adding: c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>